# 📒 Bài 3 — Mạng tích chập (CNN)

**Tuần 13**

Bài 2 đạt **82%** với MLP. Bài này dùng **cùng dữ liệu** và đạt **~96%**. Ta sẽ tìm hiểu 14 điểm chênh lệch đó đến từ đâu.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]

torch.manual_seed(42)

DATA = Path("../../../data")
if not (DATA / "hinh_khoi.npz").exists():
    DATA = Path("data")

d = np.load(DATA / "hinh_khoi.npz")
TEN_LOP = list(d["ten_lop"])

# CNN cần chiều KÊNH: (n, 28, 28) -> (n, 1, 28, 28)
X_train = torch.tensor(d["X_train"]).unsqueeze(1)
y_train = torch.tensor(d["y_train"])
X_test = torch.tensor(d["X_test"]).unsqueeze(1)
y_test = torch.tensor(d["y_test"])

print("Shape cho CNN:", tuple(X_train.shape), " <- (batch, kênh, cao, rộng)")


def tao_batch(X, y, bs, seed=0):
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(X), generator=g)
    return [(X[idx[i:i+bs]], y[idx[i:i+bs]]) for i in range(0, len(X), bs)]


def train_mot_epoch(model, batches, ham_loss, opt):
    model.train()
    tong = 0.0
    for xb, yb in batches:
        loss = ham_loss(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tong += loss.item()
    return tong / len(batches)


@torch.no_grad()
def danh_gia(model, X, y, ham_loss):
    model.eval()
    logits = model(X)
    return {"loss": ham_loss(logits, y).item(),
            "acc": (logits.argmax(1) == y).float().mean().item()}

## 1. ⚠️ Chiều KÊNH — lỗi shape đầu tiên ai cũng gặp

In [ ]:
anh_2d = torch.tensor(d["X_train"][:4])       # (4, 28, 28) — thiếu chiều kênh
conv = nn.Conv2d(1, 8, 3, padding=1)

try:
    conv(anh_2d)
except RuntimeError as e:
    print("Lỗi:", str(e)[:150])

In [ ]:
anh_4d = anh_2d.unsqueeze(1)                  # (4, 1, 28, 28)
print("Sau unsqueeze(1):", tuple(anh_4d.shape))
print("Qua conv        :", tuple(conv(anh_4d).shape))

**Quy ước shape ảnh trong PyTorch: `(batch, kênh, cao, rộng)`** — gọi là NCHW.

| Loại ảnh | Kênh |
|---|---|
| Xám | 1 |
| Màu RGB | 3 |
| Ảnh vệ tinh đa phổ | 4–13 |

> ⚠️ Thư viện ảnh (PIL, OpenCV, matplotlib) dùng `(cao, rộng, kênh)` — HWC. Chuyển đổi bằng `.permute(2, 0, 1)`. Quên permute là lỗi rất phổ biến khi đọc ảnh từ file.

## 2. ⭐ Tích chập làm gì? — nhìn tận mắt

In [ ]:
anh = X_train[1, 0]

# Ba kernel kinh điển, đặt thủ công
kernels = {
    "cạnh dọc":  [[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]],
    "cạnh ngang": [[-1, -1, -1], [0, 0, 0], [1, 1, 1]],
    "làm nét":   [[0, -1, 0], [-1, 5, -1], [0, -1, 0]],
}

fig, axes = plt.subplots(1, 4, figsize=(13, 3.4))
axes[0].imshow(anh, cmap="gray"); axes[0].set_title(f"gốc ({TEN_LOP[y_train[1]]})")
axes[0].axis("off")

for ax, (ten, k) in zip(axes[1:], kernels.items()):
    kt = torch.tensor(k, dtype=torch.float32).view(1, 1, 3, 3)
    with torch.no_grad():
        ra = torch.conv2d(anh.view(1, 1, 28, 28), kt, padding=1)
    ax.imshow(ra[0, 0], cmap="gray"); ax.set_title(ten); ax.axis("off")

plt.tight_layout(); plt.show()

Mỗi kernel là **9 con số** trượt khắp ảnh, phát hiện một loại đặc trưng.

```
   Ảnh                    Kernel 3×3            Feature map
  ┌─────────────┐         ┌────────┐          ┌───────────┐
  │ ▓ ▓ ░ ░ ░ ░ │         │-1  0  1│          │           │
  │ ▓ ▓ ░ ░ ░ ░ │    ⊛    │-1  0  1│   ──▶    │ cạnh dọc  │
  │ ▓ ▓ ░ ░ ░ ░ │         │-1  0  1│          │  ở đâu?   │
  └─────────────┘         └────────┘          └───────────┘
                       trượt khắp ảnh
```

> 🔑 **Điểm mấu chốt:** cùng 9 con số đó được dùng ở **mọi vị trí**. CNN học *"cạnh dọc trông thế nào"* **một lần** rồi áp dụng khắp nơi. Đó chính là **bất biến với dịch chuyển**.
>
> Ba kernel trên do ta đặt tay. Trong CNN thật, **model tự học** ra chúng qua gradient descent.

## 3. Công thức kích thước — phải thuộc

In [ ]:
def kich_thuoc_conv(vao, kernel, stride=1, padding=0):
    return (vao + 2 * padding - kernel) // stride + 1


print(f"{'cấu hình':<38} {'ra':>4}")
print("-" * 44)
for vao, k, s, p, ghi_chu in [
    (28, 3, 1, 1, "kernel 3, padding 1  → GIỮ NGUYÊN"),
    (28, 5, 1, 2, "kernel 5, padding 2  → GIỮ NGUYÊN"),
    (28, 3, 1, 0, "không padding        → teo 2 pixel"),
    (28, 3, 2, 1, "stride 2             → giảm nửa"),
]:
    print(f"{ghi_chu:<38} {kich_thuoc_conv(vao, k, s, p):>4}")

print("\nMẹo: padding = (kernel - 1) // 2  thì kích thước GIỮ NGUYÊN")
print("Đó là lý do bạn thấy `kernel_size=3, padding=1` ở khắp nơi.")

In [ ]:
# Cách chắc chắn nhất: cho tensor giả chạy qua rồi in shape
phan_conv = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
)

x = torch.zeros(1, 1, 28, 28)
print(f"  {'đầu vào':<22} {tuple(x.shape)}")
for lop in phan_conv:
    x = lop(x)
    print(f"  {type(lop).__name__:<22} {tuple(x.shape)}")

print(f"\n  Sau Flatten: {x.numel()} đặc trưng  ->  Linear({x.numel()}, 64)")

> 💡 **Mẹo dùng cả sự nghiệp:** khi không chắc lớp `Linear` cuối phải nhận bao nhiêu đầu vào, cho một tensor giả chạy qua phần conv rồi in `.shape`. Nhanh hơn và không thể sai so với tính nhẩm.

## 4. Xây CNN

In [ ]:
def tao_cnn(so_kenh=1, so_lop=4, kich_thuoc=28):
    k = kich_thuoc // 4                      # hai lần MaxPool2d(2)
    return nn.Sequential(
        nn.Conv2d(so_kenh, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(),
        nn.Linear(32 * k * k, 64), nn.ReLU(),
        nn.Linear(64, so_lop),
    )


torch.manual_seed(0)
cnn = tao_cnn()
print(f"Kiểm tra shape: {tuple(cnn(torch.randn(8, 1, 28, 28)).shape)}\n")

for lop in cnn:
    so = sum(p.numel() for p in lop.parameters())
    if so:
        print(f"  {type(lop).__name__:<10} {so:>10,} tham số")
print(f"  {'TỔNG':<10} {sum(p.numel() for p in cnn.parameters()):>10,}")

**Mẫu chung của mọi CNN: kích thước không gian GIẢM, số kênh TĂNG.**
```
28×28×1  →  14×14×16  →  7×7×32
```
Mạng đi từ *"nhiều vị trí, ít đặc trưng"* sang *"ít vị trí, nhiều đặc trưng"*. Lớp đầu hỏi *"có cạnh ở đây không?"*, lớp sau hỏi *"toàn bộ hình này là vật thể gì?"*

> 🔍 **Con số gây bất ngờ:** hai lớp Conv chỉ có ~4.800 tham số, nhưng lớp Linear đầu tiên có ~100.000 — chiếm **95%** toàn mạng. Phần "thông minh" rất nhẹ, phần "ngô nghê" lại nặng nhất. Các kiến trúc hiện đại (ResNet, EfficientNet) giải quyết bằng `AdaptiveAvgPool2d`.

## 5. ⭐⭐ Train CNN — so sánh trực tiếp với MLP

In [ ]:
def train_day_du(model, Xtr, Xte, so_epoch=15, lr=1e-3, bs=64):
    ham_loss = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ls = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
    t0 = time.perf_counter()

    for e in range(so_epoch):
        train_mot_epoch(model, tao_batch(Xtr, y_train, bs, seed=e), ham_loss, opt)
        tr = danh_gia(model, Xtr, y_train, ham_loss)
        va = danh_gia(model, Xte, y_test, ham_loss)
        ls["train_acc"].append(tr["acc"]); ls["train_loss"].append(tr["loss"])
        ls["val_acc"].append(va["acc"]);   ls["val_loss"].append(va["loss"])

    ls["thoi_gian"] = time.perf_counter() - t0
    return ls


torch.manual_seed(0)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(),
                    nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 4))
ls_mlp = train_day_du(mlp, X_train, X_test)
print(f"  MLP  {sum(p.numel() for p in mlp.parameters()):>8,} tham số  "
      f"acc {ls_mlp['val_acc'][-1]:.4f}  ({ls_mlp['thoi_gian']:.1f}s)")

torch.manual_seed(0)
cnn = tao_cnn()
ls_cnn = train_day_du(cnn, X_train, X_test)
print(f"  CNN  {sum(p.numel() for p in cnn.parameters()):>8,} tham số  "
      f"acc {ls_cnn['val_acc'][-1]:.4f}  ({ls_cnn['thoi_gian']:.1f}s)")

print(f"\n  Chênh lệch: {ls_cnn['val_acc'][-1] - ls_mlp['val_acc'][-1]:+.1%}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2)

for ten, ls, mau in [("MLP", ls_mlp, MAU[3]), ("CNN", ls_cnn, MAU[0])]:
    a1.plot(ls["val_acc"], lw=2, color=mau, label=ten)
    a2.plot(ls["val_loss"], lw=2, color=mau, label=ten)

a1.set_xlabel("epoch"); a1.set_ylabel("val accuracy"); a1.set_title("Độ chính xác")
a2.set_xlabel("epoch"); a2.set_ylabel("val loss"); a2.set_title("Loss")
for a in (a1, a2):
    a.legend(frameon=False)
plt.tight_layout(); plt.show()

**Số tham số gần như nhau, nhưng CNN thắng ~14 điểm.** Vì sao?

| # | Lý do | Ảnh hưởng ở đây |
|---|---|---|
| **1** | **Bất biến với dịch chuyển** — chia sẻ trọng số | Lớn nhất. Hình xuất hiện ở vị trí và góc xoay ngẫu nhiên |
| **2** | **Giữ cấu trúc không gian** | MLP flatten → *"pixel nào cạnh pixel nào"* biến mất |
| **3** | **Ít tham số hơn khi ảnh lớn** | Chưa rõ với 28×28; quyết định với 224×224 |

MLP phải học riêng *"hình tròn ở góc trên trái"*, *"hình tròn ở giữa"*, *"hình tròn ở góc dưới phải"*… như những mẫu hoàn toàn khác nhau. CNN học *"đường cong khép kín trông thế nào"* **một lần**.

## 6. Model sai ở đâu?

In [ ]:
@torch.no_grad()
def confusion(model, X, y):
    model.eval()
    p = model(X).argmax(1)
    cm = torch.zeros(4, 4, dtype=torch.int)
    for t, q in zip(y, p):
        cm[t, q] += 1
    return cm, p


fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (ten, model) in zip(axes, [("MLP", mlp), ("CNN", cnn)]):
    cm, _ = confusion(model, X_test, y_test)
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(4), TEN_LOP, rotation=30, ha="right")
    ax.set_yticks(range(4), TEN_LOP)
    ax.grid(visible=False)
    ax.set_title(f"{ten} — {torch.diagonal(cm).sum()/cm.sum():.1%}")
    for i in range(4):
        for j in range(4):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=9,
                    color="white" if cm[i, j] > cm.max() / 2 else "#222")

plt.tight_layout(); plt.show()

> Nhìn kỹ: MLP nhầm nhiều nhất ở **tròn ↔ vuông** — hai hình mà phép **xoay** làm chúng trông giống nhau nhất khi bị flatten. CNN gần như không nhầm.

## 7. CNN học được gì? — nhìn vào bộ lọc

In [ ]:
bo_loc = cnn[0].weight.detach()          # (16, 1, 3, 3)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(bo_loc[i, 0], cmap="RdBu_r", vmin=-bo_loc.abs().max(), vmax=bo_loc.abs().max())
    ax.axis("off")
plt.suptitle("16 bộ lọc 3×3 mà lớp Conv đầu tiên TỰ HỌC được", y=1.05)
plt.tight_layout(); plt.show()

In [ ]:
# Feature map: mỗi bộ lọc "nhìn thấy" gì trên một ảnh cụ thể
with torch.no_grad():
    fm = cnn[0](X_test[:1])                  # (1, 16, 28, 28)

fig, axes = plt.subplots(2, 9, figsize=(13, 3.4))
axes[0, 0].imshow(X_test[0, 0], cmap="gray")
axes[0, 0].set_title(f"gốc: {TEN_LOP[y_test[0]]}", fontsize=9)
axes[0, 0].axis("off")
axes[1, 0].axis("off")

for ax, i in zip(list(axes[0, 1:]) + list(axes[1, 1:]), range(16)):
    ax.imshow(fm[0, i], cmap="gray"); ax.axis("off")

plt.suptitle("Feature map — mỗi ô là phản hồi của một bộ lọc", y=1.05)
plt.tight_layout(); plt.show()

Mỗi feature map làm nổi bật một loại đặc trưng khác nhau: cạnh theo hướng này, vùng sáng theo hướng kia. **Không ai lập trình những bộ lọc này** — chúng xuất hiện từ gradient descent.

Trong CNN sâu (ResNet, VGG), phân cấp rất rõ:
```
lớp 1: ╱ ╲ │ ─      cạnh, góc
lớp 2: ○ □ △        hình dạng, kết cấu
lớp 3: 🚗 🐱 🏠      bộ phận vật thể
```

> 🔗 Chính vì các lớp đầu học **đặc trưng dùng chung cho mọi loại ảnh** mà **transfer learning** hoạt động — bạn giữ chúng và chỉ thay lớp cuối.

## 8. Tổng kết

| | MLP | CNN |
|---|---|---|
| Nhìn ảnh như | Vector 784 số rời rạc | Lưới 2D có cấu trúc |
| Bất biến dịch chuyển | ❌ | ✅ |
| Tham số (ảnh 224×224 màu) | ~19 triệu | ~6 triệu |
| Kết quả ở đây | 82% | **96%** |

**Ba điều phải nhớ:**
```
① Shape ảnh = (batch, kênh, cao, rộng)     — nhớ .unsqueeze(1) với ảnh xám
② padding = (kernel-1)//2  → giữ nguyên kích thước
③ Không chắc shape? Cho tensor giả chạy qua rồi in .shape
```

## ✍️ Bài tập

```bash
pytest tests/phase05/test_ex03.py -v
```